In [5]:
from pathlib import Path
import numpy as np
import ezc3d
from scipy.interpolate import interp1d

ROOT = Path(r"E:/marcel.furs/Documents/Data_Run_Walk/Data_Run_Walk")

ANGLE_CHANNELS = [
    "LFootProgressAngles", "RFootProgressAngles",
    "LHipAngles",          "RHipAngles",
    "LKneeAngles",         "RKneeAngles",
    "LAnkleAngles",        "RAnkleAngles",
    "LForeFootAngles",     "RForeFootAngles",
    "LSpineAngles",        "RSpineAngles",
    "LShoulderAngles",     "RShoulderAngles",
    "LElbowAngles",        "RElbowAngles",
    "LWristAngles",        "RWristAngles",
    "LNeckAngles",         "RNeckAngles",
    "LPelvisAngles",       "RPelvisAngles",
    "LThoraxAngles",       "RThoraxAngles",
    "LHeadAngles",         "RHeadAngles",
]

SEX_MAP = {
    "AJ026": "F", "BD004": "F", "BL025": "M", "CF027": "M", "CL007": "F",
    "DA013": "M", "DC005": "M", "GF022": "F", "GM001": "M", "GQ016": "M",
    "HN021": "F", "HS018": "F", "JS009": "F", "LA014": "F", "LD002": "M",
    "LF003": "M", "LL011": "M", "LM012": "M", "MR015": "M", "OB010": "M",
    "PC008": "F", "PQ006": "M", "RC020": "F", "RC023": "F", "RV028": "M",
    "SA017": "F", "SM019": "F", "TB030": "M", "TK029": "F", "YX024": "M"
}


TARGET_FRAMES = 100     
MIN_CYCLE_FRAMES = 50    
MAX_CYCLE_FRAMES = 200 )



def extract_subject_session(filepath):
    """
    Take id of person and take number of session in file
    """
    path_str = str(filepath)
    
    path_str = path_str.replace("\\", "/")
    parts = path_str.split("/")

    subject_id = None
    for part in parts:
        if part in SEX_MAP:
            subject_id = part
            break
    

    session_num = None
    for part in parts:
        if part.startswith("Session"):
            try:
                session_num = int(part.replace("Session", ""))
            except:
                pass
            break
    
    return subject_id, session_num


def find_right_foot_strikes(c3d_file):
    """
    Finds all of right foot strike events
    """
    c3d = ezc3d.c3d(str(c3d_file))
    

    events = c3d["parameters"]["EVENT"]
    num_events = int(events["USED"]["value"][0])
    
    if num_events == 0:
        return []
    

    labels = [str(x).strip() for x in events["LABELS"]["value"][:num_events]]
    contexts = [str(x).strip() for x in events["CONTEXTS"]["value"][:num_events]]
    times_sec = events["TIMES"]["value"][1][:num_events]
    

    fps = float(c3d["parameters"]["POINT"]["RATE"]["value"][0])
    first_frame = int(c3d["header"]["points"]["first_frame"])
    last_frame = int(c3d["header"]["points"]["last_frame"])
    num_frames = last_frame - first_frame + 1

    rfs_frames = []
    for i in range(num_events):
        label = labels[i].lower()
        context = contexts[i].lower()
        
        is_strike = "strike" in label or "contact" in label
        is_right = "right" in context
        
        if is_strike and is_right:
            time_sec = times_sec[i]
            
            frame_local = int(time_sec * fps)
            
            frame_global = int(time_sec * fps) - first_frame
            
            if 0 <= frame_local < num_frames:
                rfs_frames.append(frame_local)
            elif 0 <= frame_global < num_frames:
                rfs_frames.append(frame_global)

    return sorted(rfs_frames)


def load_angle_data(c3d_file, channel_names):
    """
    Read angles data
    
    Returns:
        numpy array shape: (num_frames, num_channels, 3)
        where 3 = axis X, Y, Z
    """
    c3d = ezc3d.c3d(str(c3d_file))
    
    all_labels = [str(x) for x in c3d["parameters"]["POINT"]["LABELS"]["value"]]
    
    for ch in channel_names:
        if ch not in all_labels:
            raise ValueError(f"Kanał {ch} nie istnieje w pliku!")
    
    channel_indices = [all_labels.index(ch) for ch in channel_names]
    
    points = c3d["data"]["points"]
    
    angle_data = points[:3, channel_indices, :] 
    
    angle_data = np.transpose(angle_data, (2, 1, 0))
    
    return angle_data


def normalize_cycle_to_100_frames(cycle_data):

    num_frames, num_channels, num_axes = cycle_data.shape
    
    normalized = np.zeros((TARGET_FRAMES, num_channels, num_axes), dtype=np.float32)
    
    old_time = np.arange(num_frames)
    new_time = np.linspace(0, num_frames - 1, TARGET_FRAMES)
    
    for ch in range(num_channels):
        for ax in range(num_axes):
            values = cycle_data[:, ch, ax]
            
            interpolator = interp1d(old_time, values, kind='linear')
            normalized[:, ch, ax] = interpolator(new_time)
    
    return normalized


def extract_cycles_from_file(c3d_file):

    subject_id, _ = extract_subject_session(c3d_file)
    
    rfs_frames = find_right_foot_strikes(c3d_file)
    
    if len(rfs_frames) < 2:
        print(f"   Za mało RFS eventów ({len(rfs_frames)}), skip")
        return []
    
    try:
        angle_data = load_angle_data(c3d_file, ANGLE_CHANNELS)
        print(f"  Załadowano: {angle_data.shape[0]} ramek × {angle_data.shape[1]} kanałów × {angle_data.shape[2]} osie")
    except Exception as e:
        print(f"   Błąd wczytywania: {e}")
        return []
    
    total_frames = angle_data.shape[0]
    
    cycles = []
    for i in range(len(rfs_frames) - 1):
        start = rfs_frames[i]
        end = rfs_frames[i + 1]
        
        if start < 0 or end >= total_frames:
            print(f"  Cykl {i+1}: poza zakresem ({start}->{end}), skip")
            continue
        
        cycle_length = end - start
        
        if cycle_length < MIN_CYCLE_FRAMES:
            print(f"  Cykl {i+1}: za krótki ({cycle_length} ramek), skip")
            continue
        
        if cycle_length > MAX_CYCLE_FRAMES:
            print(f"  Cykl {i+1}: za długi ({cycle_length} ramek), skip")
            continue

        cycle = angle_data[start:end, :, :] 
        
        if np.isnan(cycle).any():
            print(f"  Cykl {i+1}: zawiera NaN, skip")
            continue
        
        cycle_normalized = normalize_cycle_to_100_frames(cycle)
        
        cycles.append(cycle_normalized)
        print(f"  Cykl {i+1}: {cycle_length} ramek -> 100 ramek")
    
    return cycles


# MAIN PREPROCESS PREPROCESSINGU

def preprocess_dataset():
    
    pattern = "**/Overground_Walk/Walk_Comfortable/Post_Process/*.c3d"
    all_files = sorted(ROOT.rglob(pattern))
    
    print(f"\nZnaleziono {len(all_files)} plików C3D")
    
    if len(all_files) == 0:
        print("BŁĄD: brak plików")
        return
    
    all_cycles = []
    all_subjects = []
    all_sessions = []
    all_sexes = []
    
    
    for file_idx, filepath in enumerate(all_files, 1):
        print(f"\n[{file_idx}/{len(all_files)}] {filepath.name}")

        subject_id, session_num = extract_subject_session(filepath)
        
        if subject_id is None:
            print("  Nie znaleziono ID osoby, skip")
            continue
        
        if session_num is None:
            print("  Nie znaleziono numeru sesji, skip")
            continue
        
        sex = SEX_MAP.get(subject_id)
        if sex is None:
            print(f"  Brak płci dla {subject_id}, skip")
            continue
        
        print(f"  Osoba: {subject_id} | Sesja: {session_num} | Płeć: {sex}")
        
        cycles = extract_cycles_from_file(filepath)
        
        if len(cycles) == 0:
            print("  ✗ Brak poprawnych cykli")
            continue
        
        for cycle in cycles:
            all_cycles.append(cycle)
            all_subjects.append(subject_id)
            all_sessions.append(session_num)
            all_sexes.append(sex)
        
        print(f"  Dodano {len(cycles)} cykli")
    
    if len(all_cycles) == 0:
        print("\n BŁĄD: Brak danych.")
        return
    
    X = np.array(all_cycles, dtype=np.float32)  # (N, 100, 13, 3)
    subjects = np.array(all_subjects)
    sessions = np.array(all_sessions)
    sexes = np.array(all_sexes)
    
    # Flatten do (N, 100, 39)
    X_flat = X.reshape(X.shape[0], 100, -1)
    
    print("\n" + "="*80)
    print("STATYSTYKI")
    print("="*80)
    
    print(f"\nŁącznie cykli: {len(X_flat)}")
    print(f"Shape danych: {X_flat.shape}")
    print(f"  - {X_flat.shape[0]} cykli")
    print(f"  - {X_flat.shape[1]} ramek na cykl (0-100%)")
    num_joints = len(ANGLE_CHANNELS)
    print(f"  - {X_flat.shape[2]} features ({num_joints} kanałów × 3 osie)")

    
    # Statystyki per płeć
    unique_subjects = np.unique(subjects)
    print(f"\nUnikalne osoby: {len(unique_subjects)}")
    
    males = sexes == "M"
    females = sexes == "F"
    
    print(f"\nKobiety (F):")
    print(f"  - Liczba cykli: {females.sum()}")
    print(f"  - Liczba osób: {len(np.unique(subjects[females]))}")
    print(f"  - Średnia cykli/osoba: {females.sum() / len(np.unique(subjects[females])):.1f}")
    
    print(f"\nMężczyźni (M):")
    print(f"  - Liczba cykli: {males.sum()}")
    print(f"  - Liczba osób: {len(np.unique(subjects[males]))}")
    print(f"  - Średnia cykli/osoba: {males.sum() / len(np.unique(subjects[males])):.1f}")
    
    # Sprawdź kobiety
    female_subjects = np.unique(subjects[females])
    missing_f = []
    print("\nKobiety - sprawdzenie sesji:")
    for subj in sorted(female_subjects):
        s1 = ((subjects == subj) & (sessions == 1)).sum()
        s2 = ((subjects == subj) & (sessions == 2)).sum()
        status = "✓" if (s1 > 0 and s2 > 0) else "⚠️"
        print(f"  {status} {subj}: S1={s1}, S2={s2}")
        if s1 == 0 or s2 == 0:
            missing_f.append(subj)
    
    # Sprawdź mężczyzn
    male_subjects = np.unique(subjects[males])
    missing_m = []
    print("\nMężczyźni - sprawdzenie sesji:")
    for subj in sorted(male_subjects):
        s1 = ((subjects == subj) & (sessions == 1)).sum()
        s2 = ((subjects == subj) & (sessions == 2)).sum()
        status = "✓" if (s1 > 0 and s2 > 0) else "⚠️"
        print(f"  {status} {subj}: S1={s1}, S2={s2}")
        if s1 == 0 or s2 == 0:
            missing_m.append(subj)
    
    if len(missing_f) == 0:
        print("\n Wszystkie kobiety mają obie sesje")
    else:
        print(f"\n {len(missing_f)} kobiet bez jednej sesji: {missing_f}")
    
    if len(missing_m) == 0:
        print("Wszyscy mężczyźni mają obie sesje")
    else:
        print(f" {len(missing_m)} mężczyzn bez jednej sesji: {missing_m}")
    

    
    # KOBIETY
    mask_train_F = (sexes == "F") & (sessions == 1)
    mask_test_F = (sexes == "F") & (sessions == 2)
    
    X_train_F = X_flat[mask_train_F]
    X_test_F = X_flat[mask_test_F]
    subjects_train_F = subjects[mask_train_F]
    subjects_test_F = subjects[mask_test_F]
    
    # MĘŻCZYŹNI
    mask_train_M = (sexes == "M") & (sessions == 1)
    mask_test_M = (sexes == "M") & (sessions == 2)
    
    X_train_M = X_flat[mask_train_M]
    X_test_M = X_flat[mask_test_M]
    subjects_train_M = subjects[mask_train_M]
    subjects_test_M = subjects[mask_test_M]

    output_dir = ROOT / "CLEAN_ANGLES_DATASET_FULLBODY"

    output_dir.mkdir(exist_ok=True, parents=True)

    num_joints = len(ANGLE_CHANNELS)

    np.save(output_dir / "all_data_fullbody.npy", {
        "X_flat": X_flat,
        "X_3D": X,
        "subjects": subjects,
        "sessions": sessions,
        "sexes": sexes,
        "channels": ANGLE_CHANNELS,
        "num_joints": num_joints,
        "num_axes": 3
    }, allow_pickle=True)

    
    # Podzielone dane
    np.save(output_dir / "train_F_S1_fullbody.npy", X_train_F)
    np.save(output_dir / "test_F_S2_fullbody.npy", X_test_F)
    np.save(output_dir / "train_M_S1_fullbody.npy", X_train_M)
    np.save(output_dir / "test_M_S2_fullbody.npy", X_test_M)

    # Metadane
    np.save(output_dir / "subjects_train_F_fullbody.npy", subjects_train_F)
    np.save(output_dir / "subjects_test_F_fullbody.npy", subjects_test_F)
    np.save(output_dir / "subjects_train_M_fullbody.npy", subjects_train_M)
    np.save(output_dir / "subjects_test_M_fullbody.npy", subjects_test_M)

if __name__ == "__main__":
    preprocess_dataset()

START PREPROCESSINGU

Znaleziono 474 plików C3D

PRZETWARZANIE PLIKÓW

[1/474] Walk_Comfortable1.c3d
  👤 Osoba: AJ026 | Sesja: 1 | Płeć: F
  📊 Załadowano: 395 ramek × 26 kanałów × 3 osie
  ✓ Cykl 1: 105 ramek -> 100 ramek
  ✓ Cykl 2: 101 ramek -> 100 ramek
  ✓ Cykl 3: 103 ramek -> 100 ramek
  ✓ Dodano 3 cykli

[2/474] Walk_Comfortable2.c3d
  👤 Osoba: AJ026 | Sesja: 1 | Płeć: F
  📊 Załadowano: 394 ramek × 26 kanałów × 3 osie
  ✓ Cykl 1: 104 ramek -> 100 ramek
  ✓ Cykl 2: 102 ramek -> 100 ramek
  ✓ Cykl 3: 101 ramek -> 100 ramek
  ✓ Dodano 3 cykli

[3/474] Walk_Comfortable3.c3d
  👤 Osoba: AJ026 | Sesja: 1 | Płeć: F
  📊 Załadowano: 378 ramek × 26 kanałów × 3 osie
  ✓ Cykl 1: 100 ramek -> 100 ramek
  ✓ Cykl 2: 98 ramek -> 100 ramek
  ✓ Cykl 3: 100 ramek -> 100 ramek
  ✓ Dodano 3 cykli

[4/474] Walk_Comfortable4.c3d
  👤 Osoba: AJ026 | Sesja: 1 | Płeć: F
  📊 Załadowano: 332 ramek × 26 kanałów × 3 osie
  ✓ Cykl 1: 100 ramek -> 100 ramek
  ✓ Cykl 2: 100 ramek -> 100 ramek
  ✓ Dodano 2 cykli

[